In [ ]:
import spacy
from spacy.training.example import Example
from spacy.util import minibatch, compounding
import random

# Charger le modèle spaCy
nlp = spacy.load("fr_core_news_md")

# Ajouter le pipe 'ner' (Named Entity Recognition) si nécessaire
if "ner" not in nlp.pipe_names:
    ner = nlp.create_pipe("ner")
    nlp.add_pipe("ner", last=True)
else:
    ner = nlp.get_pipe("ner")


In [ ]:
TRAIN_DATA = [
    ("Je vais de Paris à Lyon demain", {"entities": [(11, 16, "LOC"), (19, 23, "LOC")]}),
    ("Mon voyage de Lille à Marseille a été annulé", {"entities": [(12, 17, "LOC"), (20, 29, "LOC")]}),
    ("Il y a un vol direct de Bruxelles à Londres", {"entities": [(20, 29, "LOC"), (32, 38, "LOC")]}),
    ("Le train de Toulouse à Montpellier partira à 9h", {"entities": [(11, 19, "LOC"), (22, 34, "LOC")]}),
    ("Un bus fait la liaison entre Bordeaux et Nantes", {"entities": [(27, 34, "LOC"), (38, 44, "LOC")]}),
    ("Nous avons pris un avion de Madrid à Barcelone", {"entities": [(27, 33, "LOC"), (36, 45, "LOC")]}),
    ("Le trajet de Berlin à Amsterdam en train est très rapide", {"entities": [(12, 18, "LOC"), (21, 30, "LOC")]}),
    ("Je pars de Genève pour aller à Zurich", {"entities": [(9, 15, "LOC"), (28, 34, "LOC")]}),
    ("La route entre Nice et Monaco est magnifique", {"entities": [(14, 18, "LOC"), (22, 28, "LOC")]}),
    ("Nous avons pris un avion de Madrid à Barcelone", {"entities": [(26, 32, "LOC"), (35, 44, "LOC")]}),
    ("Un vol direct entre Rome et Milan est disponible", {"entities": [(22, 26, "LOC"), (30, 35, "LOC")]}),
    ("Ils voyageront de Vienne à Prague la semaine prochaine", {"entities": [(17, 23, "LOC"), (26, 32, "LOC")]}),
    ("Un trajet de Paris à Berlin prend environ 8 heures", {"entities": [(12, 17, "LOC"), (20, 26, "LOC")]}),
    ("Je prends un train de Marseille à Lille demain", {"entities": [(19, 28, "LOC"), (31, 36, "LOC")]}),
    ("Un trajet en voiture de Lyon à Genève est prévu", {"entities": [(24, 28, "LOC"), (31, 37, "LOC")]}),
    ("Le bus relie Strasbourg à Bruxelles en 5 heures", {"entities": [(14, 24, "LOC"), (27, 36, "LOC")]}),
    ("Nous allons en voiture de Munich à Salzbourg", {"entities": [(23, 29, "LOC"), (32, 41, "LOC")]}),
    ("Je prendrai un avion de Tokyo à Séoul pour le voyage", {"entities": [(21, 26, "LOC"), (29, 34, "LOC")]}),
    ("Je vais de Bordeaux à Toulouse pour le week-end", {"entities": [(10, 18, "LOC"), (21, 29, "LOC")]}),
    ("Un train rapide fait la liaison entre Lyon et Dijon", {"entities": [(37, 41, "LOC"), (45, 50, "LOC")]}),
    ("Le trajet entre Madrid et Lisbonne dure environ 6 heures", {"entities": [(16, 22, "LOC"), (26, 34, "LOC")]}),
    ("Le TGV va de Paris à Strasbourg en 2 heures", {"entities": [(11, 16, "LOC"), (19, 29, "LOC")]}),
    ("Le vol de Montréal à Toronto a été retardé de deux heures", {"entities": [(9, 17, "LOC"), (20, 27, "LOC")]}),
    ("Je prends un bus de Sydney à Melbourne demain", {"entities": [(19, 25, "LOC"), (28, 37, "LOC")]}),
    ("Le trajet de Chicago à New York en train est très confortable", {"entities": [(12, 19, "LOC"), (22, 30, "LOC")]}),
    ("Je vais partir en vacances de Bruxelles à Paris la semaine prochaine", {"entities": [(27, 36, "LOC"), (39, 44, "LOC")]}),
    ("Le bus fait la liaison entre Oslo et Stockholm", {"entities": [(29, 33, "LOC"), (37, 45, "LOC")]}),
    ("Nous avons réservé un vol de Los Angeles à San Francisco", {"entities": [(27, 38, "LOC"), (41, 54, "LOC")]}),
    ("Le trajet en train de Pékin à Shanghai dure 5 heures", {"entities": [(23, 28, "LOC"), (31, 39, "LOC")]}),
    ("Un ferry relie Athènes à Santorin tous les jours", {"entities": [(15, 22, "LOC"), (25, 33, "LOC")]}),
    ("Je voyagerai de Helsinki à Copenhague la semaine prochaine", {"entities": [(15, 23, "LOC"), (26, 36, "LOC")]}),
    ("Le vol de Séoul à Tokyo dure environ deux heures", {"entities": [(9, 14, "LOC"), (17, 22, "LOC")]}),
    ("Nous partirons en bateau de Vancouver à Seattle demain", {"entities": [(28, 36, "LOC"), (39, 46, "LOC")]}),
    ("Un voyage de Rome à Athènes est prévu pour la semaine prochaine", {"entities": [(13, 17, "LOC"), (20, 27, "LOC")]}),
    ("Il y a un vol direct de Buenos Aires à Santiago demain", {"entities": [(24, 36, "LOC"), (39, 47, "LOC")]}),
    ("Le bus partira de Dublin pour aller à Belfast demain", {"entities": [(16, 22, "LOC"), (37, 44, "LOC")]}),
    ("Le train relie Milan à Florence en moins de deux heures", {"entities": [(15, 20, "LOC"), (23, 31, "LOC")]}),
    ("Un trajet de Jakarta à Bali est prévu", {"entities": [(12, 19, "LOC"), (22, 26, "LOC")]}),
     # Simples trajets avec des détails supplémentaires
    ("Je prévois de partir de Paris pour Lyon avec le train à grande vitesse.", {"entities": [(22, 27, "LOC"), (33, 37, "LOC")]}),
    ("Mon vol initial entre Lille et Marseille a été annulé à cause de la météo.", {"entities": [(19, 24, "LOC"), (27, 36, "LOC")]}),
    ("Je vais prendre un vol direct de Bruxelles à Londres demain matin.", {"entities": [(29, 38, "LOC"), (41, 47, "LOC")]}),
    ("Nous avons réservé des billets pour un trajet en train de Toulouse à Montpellier.", {"entities": [(46, 54, "LOC"), (57, 69, "LOC")]}),
    
    # Avec des transports et des détails horaires
    ("Le prochain train rapide reliant Bordeaux à Nantes partira à 8h45.", {"entities": [(34, 41, "LOC"), (44, 50, "LOC")]}),
    ("Un avion décolle de Madrid à Barcelone chaque jour à 6h du matin.", {"entities": [(19, 25, "LOC"), (28, 37, "LOC")]}),
    ("Je vais réserver un billet de bus de Berlin à Amsterdam pour vendredi prochain.", {"entities": [(31, 37, "LOC"), (40, 49, "LOC")]}),
    ("Le voyage en ferry entre Marseille et Ajaccio dure environ 4 heures.", {"entities": [(25, 34, "LOC"), (37, 44, "LOC")]}),

    # Trajets avec des contextes plus complexes
    ("Ils devaient partir de Genève et arriver à Zurich avant midi, mais il y a eu un retard.", {"entities": [(20, 26, "LOC"), (40, 46, "LOC")]}),
    ("Nous avons emprunté la route de Nice à Monaco, et c'était vraiment un magnifique trajet.", {"entities": [(27, 31, "LOC"), (34, 40, "LOC")]}),
    ("Après avoir visité Rome, je prendrai le train de Florence à Milan pour le retour.", {"entities": [(32, 36, "LOC"), (53, 61, "LOC"), (64, 69, "LOC")]}),
    ("Le prochain vol partira de Paris à New York à 22h, avec un transit à Londres.", {"entities": [(22, 27, "LOC"), (30, 38, "LOC"), (59, 65, "LOC")]}),

    # Complexité supplémentaire avec des précisions sur les circonstances
    ("Je prendrai un train de Zurich à Genève pour assister à une conférence internationale.", {"entities": [(22, 28, "LOC"), (31, 37, "LOC")]}),
    ("Le vol entre Séoul et Tokyo, initialement prévu pour 9h, a été repoussé de trois heures.", {"entities": [(13, 18, "LOC"), (21, 26, "LOC")]}),
    ("Un trajet en voiture de Los Angeles à San Francisco pourrait prendre 6 heures en fonction de la circulation.", {"entities": [(23, 34, "LOC"), (37, 50, "LOC")]}),
    ("Il y a un vol direct qui relie Buenos Aires à Santiago, mais les billets sont chers.", {"entities": [(33, 45, "LOC"), (48, 56, "LOC")]}),

    # Scénarios internationaux plus complexes
    ("Nous envisageons de faire un road trip de Montréal à Toronto, puis continuer jusqu'à Ottawa.", {"entities": [(36, 44, "LOC"), (47, 54, "LOC"), (75, 81, "LOC")]}),
    ("Je vais voyager de Tokyo à Séoul en décembre, puis prendre un vol vers Pékin.", {"entities": [(17, 22, "LOC"), (25, 30, "LOC"), (60, 65, "LOC")]}),
    ("Un long trajet en train de Munich à Salzbourg est prévu pour le week-end.", {"entities": [(27, 33, "LOC"), (36, 45, "LOC")]}),
    ("Nous avons loué une voiture pour conduire de Barcelone à Madrid et visiter les alentours.", {"entities": [(38, 47, "LOC"), (50, 56, "LOC")]}),

    # Phrases avec des circonstances temporelles
    ("Je pars de Bordeaux à Toulouse cet après-midi pour assister à un événement.", {"entities": [(9, 17, "LOC"), (20, 28, "LOC")]}),
    ("Le bus de Lyon à Dijon partira à 15h et arrivera vers 18h.", {"entities": [(10, 14, "LOC"), (17, 22, "LOC")]}),
    ("Nous avons pris un TGV de Paris à Strasbourg pour arriver avant 10h.", {"entities": [(23, 28, "LOC"), (31, 41, "LOC")]}),
    ("Le trajet en train de Madrid à Lisbonne dure environ six heures.", {"entities": [(24, 30, "LOC"), (33, 41, "LOC")]}),
    
    # Scénarios incluant plusieurs villes
    ("Nous prévoyons de partir de Rome à Athènes, avec une escale à Istanbul.", {"entities": [(27, 31, "LOC"), (34, 41, "LOC"), (61, 69, "LOC")]}),
    ("Après avoir exploré Los Angeles, nous prendrons un vol pour San Diego, puis à Las Vegas.", {"entities": [(19, 30, "LOC"), (54, 63, "LOC"), (71, 80, "LOC")]}),
    ("Je partirai de Prague pour rejoindre Vienne, puis Budapest avant de revenir à Paris.", {"entities": [(13, 19, "LOC"), (36, 42, "LOC"), (49, 57, "LOC"), (79, 84, "LOC")]}),
    ("Nous avons fait un road trip de Lisbonne à Porto, puis continué jusqu'à Madrid et Barcelone.", {"entities": [(28, 36, "LOC"), (39, 44, "LOC"), (62, 68, "LOC"), (73, 82, "LOC")]}),
]


In [33]:

# Ajouter les labels pour l'entraînement
for _, annotations in TRAIN_DATA:
    for ent in annotations.get("entities"):
        ner.add_label(ent[2])

# Désactiver les autres pipes pour entraîner uniquement le NER
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.begin_training()

    # Entraîner sur plusieurs itérations
    for itn in range(30):
        random.shuffle(TRAIN_DATA)
        losses = {}
        batches = minibatch(TRAIN_DATA, size=compounding(4.0, 32.0, 1.001))
        for batch in batches:
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                example = Example.from_dict(doc, annotations)
                nlp.update([example], drop=0.5, losses=losses)
        print(f"Iteration {itn}, Losses: {losses}")

# Étape 4 : Évaluer le modèle
# Tester avec une nouvelle phrase

doc = nlp("Je voyage de Nice à Bordeaux la semaine prochaine")
for ent in doc.ents:
    print(ent.text, ent.label_)

# Étape 5 : Sauvegarder et recharger le modèle

# Sauvegarder le modèle
nlp.to_disk("model_ner_finetuned")

# Recharger le modèle
nlp2 = spacy.load("model_ner_finetuned")

# Tester le modèle rechargé avec une nouvelle phrase
doc2 = nlp2("Je prends un train de Marseille à Lille demain")
for ent in doc2.ents:
    print(ent.text, ent.label_)

/Users/pacomedhiser/Documents/epitech/T-AIA-901-NAN_2/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Je vais prendre un vol direct de Bruxelles à Londr..." with entities "[(29, 38, 'LOC'), (41, 47, 'LOC')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/Users/pacomedhiser/Documents/epitech/T-AIA-901-NAN_2/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Après avoir exploré Los Angeles, nous prendrons un..." with entities "[(19, 30, 'LOC'), (54, 63, 'LOC'), (71, 80, 'LOC')...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/Users/pacomedhiser/Documents/epitech/T-A

Iteration 0, Losses: {'ner': np.float32(147.91872)}
Iteration 1, Losses: {'ner': np.float32(45.935165)}
Iteration 2, Losses: {'ner': np.float32(45.831356)}
Iteration 3, Losses: {'ner': np.float32(44.845387)}
Iteration 4, Losses: {'ner': np.float32(44.994923)}
Iteration 5, Losses: {'ner': np.float32(36.777756)}
Iteration 6, Losses: {'ner': np.float32(28.875109)}
Iteration 7, Losses: {'ner': np.float32(23.798435)}
Iteration 8, Losses: {'ner': np.float32(24.582869)}
Iteration 9, Losses: {'ner': np.float32(20.412275)}
Iteration 10, Losses: {'ner': np.float32(22.45868)}
Iteration 11, Losses: {'ner': np.float32(16.0236)}
Iteration 12, Losses: {'ner': np.float32(19.851677)}
Iteration 13, Losses: {'ner': np.float32(18.665564)}
Iteration 14, Losses: {'ner': np.float32(11.610699)}
Iteration 15, Losses: {'ner': np.float32(16.579363)}
Iteration 16, Losses: {'ner': np.float32(6.186201)}
Iteration 17, Losses: {'ner': np.float32(6.727184)}
Iteration 18, Losses: {'ner': np.float32(5.976395)}
Iteration